# FraudLens: running our fraud detection prototype

Our final aim is a prototype that takes a transaction dataset and predicts fraud. Our experiments showed that the useful signal changes between datasets: the time gap on Sparkov, the merchant location on TabFormer. So the prototype builds both kinds of features and uses the same leak-free protocol as our notebooks.

**FraudLens** is a small web app (built with Streamlit). In five steps it:
1. loads a transaction table (CSV);
2. asks which column plays which role (label, time, card, amount, category, age, gender, home and merchant location);
3. checks each raw column for leakage, splits the data by time, trains a Random Forest with our settings, picks the F2 threshold on validation, and shows PR-AUC, F2, MCC, the precision-recall curve, feature importance and alert budgets on the test part;
4. lists the riskiest transactions with short reasons (for example "very soon after the card's previous transaction", "merchant abroad");
5. scores new transactions with the trained model, and lets us download the model and the scores.

Everything for the app is in the `app/` folder: the app (`fraud_app.py`), its helper functions (`prototype.py`), this notebook and `requirements.txt`. The folder runs on its own and does not depend on the experiment code in `src/`. In this notebook we prepare three demo files and start the app inside Colab.

## 1. Environment setup
We type credentials in at runtime, so they are never saved in the notebook.

In [ ]:
# 1.1 Clone the project repository and install Streamlit
import os
from getpass import getpass

github_token = getpass('Paste your GitHub token: ').strip()
repo_url = f'https://{github_token}@github.com/subbareddypullalacheruvu/Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-.git'
!git clone {repo_url}
%cd Major-Project-Robust-Fraud-Detection-with-Spatial-Temporal-Features-and-Class-Imbalance-Learning-
!pip install -q -r app/requirements.txt

In [ ]:
# 1.2 Download the demo datasets from Kaggle
os.environ['KAGGLE_USERNAME'] = input('Kaggle username: ')
os.environ['KAGGLE_KEY'] = getpass('Kaggle API key: ')

!kaggle datasets download -d kartik2112/fraud-detection -p /content/data/sparkov --unzip
!kaggle datasets download -d ealtman2019/credit-card-transactions -p /content/data/tabformer --unzip
!kaggle datasets download -d mohankrishnathalla/credit-card-fraud-detection-dataset-2026 -p /content/data/cc2026 --unzip

## 2. Preparing the demo files
The full datasets are large, so we make smaller demo files in `app/demo_data/`. We keep every transaction of a random sample of cards (or users), so the time gaps stay correct.
- **Sparkov:** 400 cards from the training and test files together.
- **TabFormer:** 150 users from 2015 onwards. We join the users file to add the home state and the date of birth, and we combine the date and time columns into one column, so the app can read it.
- **Credit Card Fraud 2026:** the whole file (100,000 rows).

In [ ]:
# 2.1 Sparkov demo file
import glob
import numpy as np
import pandas as pd

os.makedirs('app/demo_data', exist_ok=True)
rng = np.random.default_rng(42)

sparkov = pd.concat([pd.read_csv(f) for f in glob.glob('/content/data/sparkov/fraud*.csv')], ignore_index=True)
sparkov = sparkov.drop(columns=[c for c in sparkov.columns if c.startswith('Unnamed')])
cards = rng.choice(sparkov['cc_num'].unique(), size=400, replace=False)
sparkov_demo = sparkov[sparkov['cc_num'].isin(cards)].sort_values('trans_date_trans_time')
sparkov_demo.to_csv('app/demo_data/sparkov_sample.csv', index=False)
print(f"Sparkov demo: {len(sparkov_demo):,} rows, {sparkov_demo['is_fraud'].sum():,} fraud")

In [ ]:
# 2.2 TabFormer demo file
users = pd.read_csv('/content/data/tabformer/sd254_users.csv')
users['User'] = np.arange(len(users))
chosen = set(rng.choice(users['User'], size=150, replace=False).tolist())

parts = []
for chunk in pd.read_csv('/content/data/tabformer/credit_card_transactions-ibm_v2.csv', chunksize=2_000_000):
    parts.append(chunk[chunk['User'].isin(chosen) & (chunk['Year'] >= 2015)])
tab = pd.concat(parts, ignore_index=True)

tab['transaction_time'] = (pd.to_datetime(dict(year=tab['Year'], month=tab['Month'], day=tab['Day']))
                           + pd.to_timedelta(tab['Time'] + ':00')).astype(str)
tab['card_id'] = tab['User'].astype(str) + '_' + tab['Card'].astype(str)
users['dob'] = pd.to_datetime(dict(year=users['Birth Year'], month=users['Birth Month'], day=1)).astype(str)
tab = tab.merge(users[['User', 'dob', 'Gender', 'State']], on='User', how='left')
tab = tab[['transaction_time', 'card_id', 'Amount', 'MCC', 'Use Chip', 'Merchant City', 'Merchant State',
           'State', 'dob', 'Gender', 'Is Fraud?']].sort_values('transaction_time')
tab.to_csv('app/demo_data/tabformer_sample.csv', index=False)
print(f"TabFormer demo: {len(tab):,} rows, {(tab['Is Fraud?'] == 'Yes').sum():,} fraud")

In [ ]:
# 2.3 Credit Card Fraud 2026 demo file
cc = pd.read_csv(glob.glob('/content/data/cc2026/*.csv')[0])
cc.to_csv('app/demo_data/cc2026.csv', index=False)
print(f"Credit Card Fraud 2026: {len(cc):,} rows, {cc['is_fraud'].sum():,} fraud")

## 3. Starting the app
We start Streamlit in the background and open it in a Colab window. If the window does not open, cell 3.2 gives a public link instead.

**Columns to choose in step 2 of the app** (most are filled in automatically):

| Demo file | Label | Time | Card | Amount | Category | Customer | Location |
|---|---|---|---|---|---|---|---|
| `sparkov_sample.csv` | `is_fraud` | `trans_date_trans_time` | `cc_num` | `amt` | `category` | `dob`, `gender` | `lat`, `long`, `merch_lat`, `merch_long` |
| `tabformer_sample.csv` | `Is Fraud?` | `transaction_time` | `card_id` | `Amount` | `MCC` | `dob`, `Gender` | home state `State`, merchant state `Merchant State` |
| `cc2026.csv` | `is_fraud` | `transaction_datetime` | `card_id` | `transaction_amount` | `merchant_category` | `customer_age`, `customer_gender` | merchant state or country `merchant_country` (home state left as *(none)*) |

Sparkov has merchant coordinates but no merchant state, so the app builds the distance from home instead of the state features. Credit Card Fraud 2026 gives the merchant country, so the app builds only the abroad feature there.

In [ ]:
# 3.1 Start the app and open it in a Colab window
!nohup streamlit run app/fraud_app.py --server.port 8501 --server.headless true --server.maxUploadSize 1000 > streamlit.log 2>&1 &
import time
time.sleep(8)
from google.colab import output
output.serve_kernel_port_as_window(8501)

In [ ]:
# 3.2 Optional: a public link, if the Colab window does not open
# The page asks for a password: it is the IP address printed by the first line.
!curl -s ipv4.icanhazip.com
!npx --yes localtunnel --port 8501

## 4. What we found with the demo files